# VAE Math Foundations

This companion notebook explains the math ideas behind `notebooks/05_vae.ipynb`. It is not a second VAE training notebook. The goal is to understand why the VAE code uses reconstruction loss, KL loss, `mu`, `logvar`, and the reparameterization trick.

## Learning purpose

Learn how VAE probability ideas make a latent space sampleable and how those ideas become the loss terms used in PyTorch code.

## Reference path

This notebook is designed to sit beside the practical notebook:

```text
notebooks/05_vae.ipynb
```

We will move slowly from ordinary autoencoders to the VAE loss function, then map the math back to the existing `loss_function()` implementation.


## 1. Ordinary autoencoder math

A regular autoencoder has two learned parts: an **encoder** and a **decoder**. The encoder compresses an input image into a smaller hidden representation. The decoder tries to rebuild the original image from that hidden representation.

```text
image x → encoder f(x) → latent code z → decoder g(z) → reconstruction x_hat
```

Meaning of the symbols:

- `x` is the original input image, such as one MNIST digit.
- `f(x)` is the encoder's output.
- `z` is the **latent code**, a smaller learned summary of the image.
- `g(z)` is the decoder's output.
- `x_hat` is the reconstructed version of the original image.

The important constraint is the **bottleneck**. The bottleneck is the smaller middle space where the model has fewer numbers than the original image. For MNIST, an input image has `28 × 28 = 784` pixel values. If the latent code has only 20 numbers, the model cannot simply store every pixel.

That pressure is useful. To reconstruct the image from fewer numbers, the model has to learn patterns that help rebuild many similar images, such as digit shape, stroke angle, loop size, and thickness. This is why autoencoders can be useful for compression, denoising, anomaly detection, and feature extraction.


### Reconstruction objective

A basic autoencoder trains by making the reconstruction `x_hat` close to the original input `x`.

```text
reconstruction loss = difference between x and x_hat
```

If the rebuilt image looks like the input image, the reconstruction loss is small. If the rebuilt image misses important pixels or changes the digit identity, the reconstruction loss is large.

This is the first half of the VAE story. A VAE still cares about reconstruction quality, but it changes the middle of the model from one fixed code into a small probability region.

```text
regular autoencoder:
x → one fixed latent code z → x_hat

variational autoencoder:
x → latent distribution q(z|x) → sampled latent code z → x_hat
```

A good plain-language explanation of the bottleneck is:

> The bottleneck forces the encoder to store only the most useful information in a smaller latent code, because the decoder must reconstruct the original image from that limited code.


## 2. Why a VAE changes the autoencoder

A regular autoencoder can reconstruct real inputs well because each latent code came from a real image. The decoder practices on codes produced by the encoder during training.

Generation asks for something harder:

```text
random latent code z → decoder → new image?
```

A regular autoencoder is not forced to make every random point in latent space meaningful. The encoder might place real image codes in separated islands. The empty regions between those islands are **latent gaps**: places where the decoder did not learn a reliable meaning.

If a random latent point lands in one of those gaps, it may correspond to a hidden representation the decoder has not learned from. The output can be blurry, broken, or not digit-like, even if reconstructions of real images look good.

A VAE changes the middle of the model so the encoder predicts a distribution instead of one fixed code.

```text
regular autoencoder:
x → fixed latent code z

variational autoencoder:
x → latent distribution q(z|x) → sampled latent code z
```

This matters because useful generation needs random `z` values to land in regions the decoder understands. The VAE will use a simple prior distribution, usually `N(0, I)`, as the shape we want latent clouds to stay near. That pressure makes the latent space more organized and sampleable.


## 3. Probability basics for VAE latent clouds

A VAE uses probability language because it does not encode an image as one exact hidden point. It encodes an image as a small region of likely hidden points, then samples one point from that region.

Important terms:

- A **random variable** is a value that is sampled instead of fixed. In a VAE, each latent number can be treated as a random variable.
- A **distribution** describes which values are likely and which values are unlikely.
- The **mean** is the center of the distribution. In VAE code, this is `mu`.
- The **variance** measures spread. Larger variance means samples are more spread out.
- The **standard deviation** is another spread measure, written as `std`. It is the square root of variance.
- The **standard normal distribution**, written `N(0, I)`, is centered at 0 with unit spread in each latent dimension.

A beginner-friendly picture is a **latent cloud**. The mean is the center of the cloud. The standard deviation is how wide the cloud is. Sampling means picking one point from that cloud.


### Notation checkpoint: symbols as names for ideas

Math notation is a compact naming system. In this notebook, each symbol should connect back to a plain-language idea and a PyTorch code name.

| Symbol | How to read it | Plain meaning | Code connection |
| --- | --- | --- | --- |
| `x` | x | the original input image | `images` or `x` |
| `x_hat` or `\hat{x}` | x-hat | the reconstructed version of the input | `recon_x` or `recon_batch` |
| `z` | z | one sampled latent code | `z` |
| `mu` or `μ` | mu | the center of the latent cloud | `mu` from `fc21` |
| `sigma` or `σ` | sigma | the standard deviation, or spread, of the latent cloud | `std` |
| `logvar` | log variance | the neural network's stored spread output before conversion to `std` | `logvar` from `fc22` |
| `q(z\|x)` | q of z given x | the encoder's distribution of likely latent codes for this input image | described by `mu` and `logvar` |
| `p(z)` | p of z | the simple prior distribution we want to sample from later | usually `N(0, I)` or `torch.randn(...)` |

The vertical bar in `q(z\|x)` means **given** or **conditioned on**. So `q(z\|x)` does not mean `z` divided by `x`; it means: for this input image `x`, what latent-code values `z` does the encoder think are likely?


## 4. VAE encoder output: `mu` and `logvar`

In the practical notebook, the encoder has two output heads:

```python
self.fc21 = nn.Linear(400, latent_dim)
self.fc22 = nn.Linear(400, latent_dim)
```

The first head outputs `mu`, the center of the latent cloud. The second head outputs ordinary neural-network numbers that the code names `logvar`.

The key idea is:

> `fc22` does not output variance directly. It outputs unconstrained numbers that we treat as log-variance.

This is useful because a neural network can output any real number: negative, zero, or positive. But variance cannot be negative, because a negative spread does not make sense. By predicting `logvar`, the model can output any real number first, and the code can convert it into a positive spread later.

The practical notebook does that conversion here:

```python
std = torch.exp(0.5 * logvar)
```

Why this works:

```text
logvar = log(variance)
variance = exp(logvar)
std = sqrt(variance) = exp(0.5 * logvar)
```

So `mu` moves the cloud, and `logvar` becomes the cloud's width after `exp(...)`. Training teaches both outputs to become useful for reconstruction and sampling.


In [1]:
# Tiny numeric example: convert logvar into variance and standard deviation.

import math

for logvar in [-2.0, 0.0, 2.0]:
    variance = math.exp(logvar)
    std = math.exp(0.5 * logvar)
    print(f"logvar={logvar:>4.1f}  variance={variance:>5.2f}  std={std:>5.2f}")


logvar=-2.0  variance= 0.14  std= 0.37
logvar= 0.0  variance= 1.00  std= 1.00
logvar= 2.0  variance= 7.39  std= 2.72
